# Meta 1 ECAC - Testing Feature Selection Methods

In [2]:
# Importation of all Frameworks and Tools needed

import pandas as pd
import numpy as np
import scipy as sc

import seaborn as sns
import matplotlib.pyplot as plt

### Creation of a rotine to create the dataframe with all data from https://github.com/spl-icsforth/FORTH_TRACE_DATASET/

In [3]:
def load_csv(url, participant, device):
    columns = [
        "DeviceId",
        "Acc_x", "Acc_y", "Acc_z",
        "Gyro_x", "Gyro_y", "Gyro_z",
        "Mag_x", "Mag_y", "Mag_z",
        "Timestamp", "Activity"
    ]
    
    df = pd.read_csv(url, header=None, names=columns)
    
    df["Participant"] = participant
    
    return df

# Parameters
individuals = 15
devices = 5

df_all = []

for part in range(individuals):
    for device in range(1, devices + 1):
        url = f"https://raw.githubusercontent.com/spl-icsforth/FORTH_TRACE_DATASET/master/part{part}/part{part}dev{device}.csv"
        try:
            df_device = load_csv(url, part, device)
            df_all.append(df_device)
            print(f"Loaded part{part}dev{device}.csv ({len(df_device)} rows)")
        except Exception as e:
            print(f"Error loading part{part}dev{device}: {e}")

# Combine all small pieces
df = pd.concat(df_all, axis=0, ignore_index=True)

print("Final dataset shape:", df.shape)
df.head()

Loaded part0dev1.csv (53120 rows)
Loaded part0dev2.csv (52864 rows)
Loaded part0dev3.csv (53120 rows)
Loaded part0dev4.csv (53120 rows)
Loaded part0dev5.csv (53120 rows)
Loaded part1dev1.csv (53376 rows)
Loaded part1dev2.csv (53376 rows)
Loaded part1dev3.csv (52992 rows)
Loaded part1dev4.csv (53376 rows)
Loaded part1dev5.csv (52096 rows)
Loaded part2dev1.csv (53248 rows)
Loaded part2dev2.csv (53248 rows)
Loaded part2dev3.csv (53248 rows)
Loaded part2dev4.csv (53248 rows)
Loaded part2dev5.csv (53248 rows)
Loaded part3dev1.csv (53268 rows)
Loaded part3dev2.csv (53247 rows)
Loaded part3dev3.csv (53291 rows)
Loaded part3dev4.csv (53199 rows)
Loaded part3dev5.csv (53235 rows)
Loaded part4dev1.csv (53120 rows)
Loaded part4dev2.csv (52992 rows)
Loaded part4dev3.csv (36352 rows)
Loaded part4dev4.csv (53120 rows)
Loaded part4dev5.csv (53120 rows)
Loaded part5dev1.csv (52861 rows)
Loaded part5dev2.csv (52864 rows)
Loaded part5dev3.csv (52352 rows)
Loaded part5dev4.csv (52864 rows)
Loaded part5de

,DeviceId,Acc_x,Acc_y,Acc_z,Gyro_x,Gyro_y,Gyro_z,Mag_x,Mag_y,Mag_z,Timestamp,Activity,Participant
0,1,-1.8650,9.3890,2.5812,-1.14180,-1.18560,0.84998,-0.34476,0.59839,1.0134,505.89,1,0
1,1,-1.7963,9.3742,2.4460,-1.56180,-0.66165,0.59730,-0.34274,0.57631,1.0000,525.42,1,0
2,1,-1.8696,9.3000,2.3514,-1.18770,-1.28410,0.14212,-0.34476,0.59639,1.0156,544.95,1,0
3,1,-1.7961,9.3624,2.4584,-0.58399,-2.03340,0.42912,-0.32863,0.62249,1.0156,564.48,1,0
4,1,-1.6768,9.3506,2.4685,-0.37050,-1.36470,0.37194,-0.33669,0.62048,1.0245,584.01,1,0


In [4]:
def add_magnitude_columns(df):
    df["Acc_module"] = np.sqrt(df["Acc_x"]**2 + df["Acc_y"]**2 + df["Acc_z"]**2)
    df["Gyro_module"] = np.sqrt(df["Gyro_x"]**2 + df["Gyro_y"]**2 + df["Gyro_z"]**2)
    df["Mag_module"] = np.sqrt(df["Mag_x"]**2 + df["Mag_y"]**2 + df["Mag_z"]**2)
    return df

df = add_magnitude_columns(df)


In [5]:
df

,DeviceId,Acc_x,Acc_y,Acc_z,Gyro_x,Gyro_y,Gyro_z,Mag_x,Mag_y,Mag_z,Timestamp,Activity,Participant,Acc_module,Gyro_module,Mag_module
0,1,-1.865000,9.3890,2.5812,-1.14180,-1.18560,0.84998,-0.34476,0.59839,1.01340,505.89,1,0,9.914340,1.852517,1.226340
1,1,-1.796300,9.3742,2.4460,-1.56180,-0.66165,0.59730,-0.34274,0.57631,1.00000,525.42,1,0,9.853184,1.798268,1.203995
2,1,-1.869600,9.3000,2.3514,-1.18770,-1.28410,0.14212,-0.34476,0.59639,1.01560,544.95,1,0,9.773151,1.754919,1.227185
3,1,-1.796100,9.3624,2.4584,-0.58399,-2.03340,0.42912,-0.32863,0.62249,1.01560,564.48,1,0,9.845011,2.158681,1.235692
4,1,-1.676800,9.3506,2.4685,-0.37050,-1.36470,0.37194,-0.33669,0.62048,1.02450,584.01,1,0,9.815237,1.462196,1.244169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3930868,5,0.042622,9.5341,-2.0574,0.21654,-0.36095,0.52635,-0.19192,0.47106,-0.68913,1042000.00,1,14,9.753654,0.673958,0.856523
3930869,5,0.030433,9.5218,-2.0691,0.32055,-0.11574,0.38508,-0.23232,0.50100,-0.68043,1042000.00,1,14,9.744064,0.514232,0.876332
3930870,5,0.018118,9.5337,-2.0691,0.23099,-0.28532,0.40054,-0.19798,0.49900,-0.67174,1042000.00,1,14,9.755662,0.543320,0.859902
3930871,5,0.017996,9.5458,-2.0693,0.35203,-0.22221,0.45125,-0.20202,0.46906,-0.67391,1042100.00,1,14,9.767529,0.613945,0.845567


In [13]:
from sklearn.preprocessing import StandardScaler
from skfeature.function.similarity_based import fisher_score

# =============================================
# 1. Load Dataset
# =============================================
df = pd.read_csv("feature_set_v.csv")

# Define features and labels
X = df.drop(columns=['Activity']).values
y = df['Activity'].values
feature_names = df.drop(columns=['Activity']).columns

# =============================================
# 2. Standardize Features
# =============================================
# Standardization is essential for both Fisher Score and ReliefF
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =============================================
# 3. Fisher Score
# =============================================
print("🔹 Running Fisher Score Feature Selection...")

# Compute Fisher Scores
fisher_scores = fisher_score.fisher_score(X_scaled, y)

# Rank features (descending order)
fisher_idx = np.argsort(fisher_scores)[::-1]

# Normalize Fisher Scores (optional, for comparison with ReliefF)
fisher_scores_norm = (fisher_scores - fisher_scores.min()) / (fisher_scores.max() - fisher_scores.min())

# Build results DataFrame
fisher_results = pd.DataFrame({
    'Feature': feature_names[fisher_idx],
    'Fisher Score': fisher_scores[fisher_idx],
    'Fisher Score (Normalized)': fisher_scores_norm[fisher_idx]
}).reset_index(drop=True)

top_fisher = fisher_results.head(10)
print("\nTop 10 Fisher Score Features:")
print(top_fisher)

🔹 Running Fisher Score Feature Selection...

Top 10 Fisher Score Features:
      Feature  Fisher Score  Fisher Score (Normalized)
0          AI           177                   1.000000
1       DF_mx           176                   0.994350
2     gy_mean           175                   0.988701
3     my_skew           174                   0.983051
4   mz_median           173                   0.977401
5    AVH_bin3           172                   0.971751
6  corr_gy_mx           171                   0.966102
7       EN_ax           170                   0.960452
8     ay_skew           169                   0.954802
9      gz_std           168                   0.949153


In [14]:
from skrebate import ReliefF

# =============================================
# 4. ReliefF
# =============================================
print("\n🔹 Running ReliefF Feature Selection...")

# Initialize ReliefF
# Adjust n_neighbors depending on dataset size (10–30 is typical)
relieff = ReliefF(n_neighbors=30, n_features_to_select=X_scaled.shape[1])

# Fit ReliefF
relieff.fit(X_scaled, y)

# Get feature importances
relief_scores = relieff.feature_importances_

# Rank features (descending order)
relief_idx = np.argsort(relief_scores)[::-1]

# Build results DataFrame
relief_results = pd.DataFrame({
    'Feature': feature_names[relief_idx],
    'ReliefF Score': relief_scores[relief_idx]
}).reset_index(drop=True)

top_relief = relief_results.head(10)
print("\nTop 10 ReliefF Features:")
print(top_relief)


🔹 Running ReliefF Feature Selection...

Top 10 ReliefF Features:
      Feature  ReliefF Score
0      mx_rms       0.118040
1       EN_mx       0.102085
2   mx_median       0.093085
3    AVH_bin1       0.080090
4      mz_rms       0.078888
5     mx_mean       0.078555
6  corr_mx_mz       0.076905
7      az_rms       0.076176
8  corr_gx_gz       0.071997
9   mz_median       0.070060
